# Repository Code Output Formatter & Explorer

This notebook helps you understand, format, and work with code output from the NuSyQ-Hub repository effectively. It provides tools for:

1. **Parsing and formatting** JSON, CSV, and text outputs from repository scripts
2. **Syntax highlighting** for better code readability
3. **Interactive exploration** of repository structure and analysis results
4. **Export capabilities** for reports and documentation

## Quick Start
Run the cells below sequentially to set up your environment and start exploring repository outputs.

## 1. Import Required Libraries

Import essential libraries for handling code output formatting and repository exploration.

In [2]:
# Essential libraries for code output handling
# For interactive widgets
import json
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, Markdown
from ipywidgets import interactive

# For syntax highlighting and display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import get_lexer_by_name, guess_lexer

# Repository-specific imports
sys.path.append(str(Path.cwd().parent / "src"))
try:
    from utils.generate_structure_tree import build_tree, extract_context
    print("✅ Repository structure generator imported successfully")
except ImportError as e:
    print(f"⚠️ Could not import structure generator: {e}")

print("📚 All libraries imported successfully!")

✅ Repository structure generator imported successfully
📚 All libraries imported successfully!


## 2. Read and Parse Code Output

Load and parse different types of output files from repository analysis and diagnostics.

In [3]:
def load_json_file(file_path):
    """Load and parse JSON files with error handling."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error in {file_path}: {e}")
        return None

def load_log_file(file_path, max_lines=100):
    """Load log files with line limiting."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
            if len(lines) > max_lines:
                print(f"📝 Showing first {max_lines} lines of {len(lines)} total")
                return lines[:max_lines]
            return lines
    except FileNotFoundError:
        print(f"❌ Log file not found: {file_path}")
        return []

# Load key repository files
repo_root = Path.cwd().parent
key_files = {
    "component_index": repo_root / "KILO_COMPONENT_INDEX.json",
    "dependency_map": repo_root / "ULTIMATE_DEPENDENCY_MAP.json",
    "import_health": repo_root / "import_health_check.log",
    "file_audit": repo_root / "file_organization_audit.log",
    "dependency_report": repo_root / "DEPENDENCY_MAPPING_REPORT.md"
}

print("🔍 Available repository analysis files:")
for name, path in key_files.items():
    exists = "✅" if path.exists() else "❌"
    print(f"  {exists} {name}: {path.name}")

🔍 Available repository analysis files:
  ✅ component_index: KILO_COMPONENT_INDEX.json
  ✅ dependency_map: ULTIMATE_DEPENDENCY_MAP.json
  ✅ import_health: import_health_check.log
  ✅ file_audit: file_organization_audit.log
  ✅ dependency_report: DEPENDENCY_MAPPING_REPORT.md


## 3. Format JSON and Structured Data

Pretty-print JSON data and format nested dictionaries for better readability.

In [4]:
def format_json_output(data, indent=2):
    """Format JSON data with proper indentation."""
    if data is None:
        return "No data to format"
    return json.dumps(data, indent=indent, ensure_ascii=False)

def display_json_summary(data, max_keys=10):
    """Display a summary of JSON structure."""
    if not isinstance(data, dict):
        return f"Data type: {type(data).__name__}"
    
    summary = []
    summary.append(f"📊 JSON Summary ({len(data)} top-level keys)")
    
    for i, (key, value) in enumerate(data.items()):
        if i >= max_keys:
            summary.append(f"   ... and {len(data) - max_keys} more keys")
            break
        
        if isinstance(value, dict):
            summary.append(f"   🗂️  {key}: dict with {len(value)} items")
        elif isinstance(value, list):
            summary.append(f"   📋 {key}: list with {len(value)} items")
        else:
            summary.append(f"   📄 {key}: {type(value).__name__}")
    
    return "\n".join(summary)

def create_dataframe_from_json(data, key_path=None):
    """Convert JSON data to pandas DataFrame when possible."""
    try:
        if key_path and isinstance(data, dict):
            # Navigate to nested data
            current = data
            for key in key_path.split("."):
                current = current[key]
            data = current
        
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            return df
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
            return df
        else:
            return None
    except Exception as e:
        print(f"Could not convert to DataFrame: {e}")
        return None

print("🎨 JSON formatting functions ready!")

🎨 JSON formatting functions ready!


## 4. Display Code with Syntax Highlighting

Use pygments and IPython.display to show code with proper syntax highlighting and formatting.

In [5]:
def highlight_code(code, language="python", style="default"):
    """Display code with syntax highlighting."""
    try:
        if language == "auto":
            lexer = guess_lexer(code)
        else:
            lexer = get_lexer_by_name(language)
        
        formatter = HtmlFormatter(style=style, cssclass="highlight")
        highlighted = highlight(code, lexer, formatter)
        
        # Add CSS styling
        css = formatter.get_style_defs(".highlight")
        html = f"<style>{css}</style>{highlighted}"
        
        return HTML(html)
    except Exception as e:
        print(f"Highlighting failed: {e}")
        return HTML(f"<pre>{code}</pre>")

def display_file_content(file_path, language="auto", max_lines=50):
    """Display file content with syntax highlighting."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        
        lines = content.split("\n")
        if len(lines) > max_lines:
            content = "\n".join(lines[:max_lines])
            content += f"\n\n... ({len(lines) - max_lines} more lines truncated)"
        
        if language == "auto":
            # Try to guess language from file extension
            ext = Path(file_path).suffix.lower()
            language_map = {
                ".py": "python", ".js": "javascript", ".json": "json",
                ".md": "markdown", ".yml": "yaml", ".yaml": "yaml",
                ".ps1": "powershell", ".sh": "bash", ".txt": "text"
            }
            language = language_map.get(ext, "text")
        
        print(f"📄 {Path(file_path).name} ({language})")
        return highlight_code(content, language)
    
    except FileNotFoundError:
        return HTML(f"<p>❌ File not found: {file_path}</p>")
    except Exception as e:
        return HTML(f"<p>❌ Error reading file: {e}</p>")

def display_repository_structure():
    """Generate and display the repository structure tree."""
    try:
        if "build_tree" in globals():
            # Generate structure and read the output file
            build_tree(Path.cwd().parent, max_depth=3)
            
            # Read the generated markdown file
            output_file = Path.cwd().parent / "REPO_STRUCTURE.md"
            if output_file.exists():
                with open(output_file, "r", encoding="utf-8") as f:
                    content = f.read()
                return Markdown(content)
            else:
                return HTML("<p>⚠️ Structure file was not generated</p>")
        else:
            return HTML("<p>⚠️ Repository structure generator not available</p>")
    except Exception as e:
        return HTML(f"<p>❌ Error generating structure: {e}</p>")

print("🎯 Syntax highlighting functions ready!")

🎯 Syntax highlighting functions ready!


## 5. Interactive Code Exploration

Create interactive widgets to explore code structure, search through output, and filter results dynamically.

In [6]:
def create_file_explorer():
    """Create an interactive file explorer widget."""
    repo_files = []
    repo_root = Path.cwd().parent
    for root, dirs, files in os.walk(repo_root):
        # Skip common ignore directories
        dirs[:] = [d for d in dirs if d not in {".venv", "__pycache__", ".git", ".pytest_cache", ".snapshots"}]
        for file in files:
            if file.endswith((".py", ".json", ".md", ".txt", ".log", ".yml", ".yaml")):
                repo_files.append(os.path.join(root, file))
    
    file_dropdown = widgets.Dropdown(
        options=repo_files,
        description="File:",
        style={"description_width": "initial"}
    )
    
    language_dropdown = widgets.Dropdown(
        options=["auto", "python", "json", "markdown", "yaml", "text"],
        value="auto",
        description="Language:"
    )
    
    max_lines_slider = widgets.IntSlider(
        value=50,
        min=10,
        max=200,
        step=10,
        description="Max lines:"
    )
    
    def display_selected_file(file_path, language, max_lines):
        return display_file_content(file_path, language, max_lines)
    
    return interactive(display_selected_file, 
                      file_path=file_dropdown, 
                      language=language_dropdown,
                      max_lines=max_lines_slider)

def create_json_explorer():
    """Create an interactive JSON data explorer."""
    repo_root = Path.cwd().parent
    json_files = []
    
    # Find JSON files in key locations
    key_locations = [
        repo_root / "KILO_COMPONENT_INDEX.json",
        repo_root / "ULTIMATE_DEPENDENCY_MAP.json"
    ]
    
    json_files = [f for f in key_locations if f.exists()]
    
    # Also search for other JSON files
    for root, dirs, files in os.walk(repo_root):
        dirs[:] = [d for d in dirs if d not in {".venv", "__pycache__", ".git"}]
        for file in files:
            if file.endswith(".json"):
                file_path = Path(root) / file
                if file_path not in json_files:
                    json_files.append(file_path)
    
    if not json_files:
        return widgets.HTML("No JSON files found in repository")
    
    file_dropdown = widgets.Dropdown(
        options=[(f.name, f) for f in json_files[:20]],  # Limit to first 20 files
        description="JSON File:"
    )
    
    def explore_json(file_path):
        data = load_json_file(file_path)
        if data is None:
            return widgets.HTML("Could not load JSON data")
        
        summary = display_json_summary(data)
        formatted = format_json_output(data, indent=2)
        
        # Create tabs for different views
        summary_widget = widgets.HTML(f"<pre>{summary}</pre>")
        formatted_widget = widgets.HTML(f"<pre>{formatted[:5000]}{'...' if len(formatted) > 5000 else ''}</pre>")
        
        tab = widgets.Tab(children=[summary_widget, formatted_widget])
        tab.set_title(0, "Summary")
        tab.set_title(1, "Raw JSON")
        
        return tab
    
    return interactive(explore_json, file_path=file_dropdown)

print("🎮 Interactive exploration widgets ready!")

🎮 Interactive exploration widgets ready!


## 6. Export and Save Results

Save formatted output to files, generate reports, and create reusable code snippets from the analysis.

In [7]:
def save_formatted_output(data, output_file, format_type="json"):
    """Save formatted data to file."""
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    try:
        if format_type == "json":
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        elif format_type == "csv" and isinstance(data, pd.DataFrame):
            data.to_csv(output_path, index=False)
        elif format_type == "markdown":
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(str(data))
        else:
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(str(data))
        
        print(f"✅ Saved to {output_path}")
        return output_path
    except Exception as e:
        print(f"❌ Error saving file: {e}")
        return None

def generate_analysis_report():
    """Generate a comprehensive analysis report."""
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    report_data = {
        "timestamp": timestamp,
        "repository": "NuSyQ-Hub",
        "analysis_files": {},
        "summary": {}
    }
    
    # Load and summarize key files
    for name, path in key_files.items():
        if path.exists():
            if path.suffix == ".json":
                data = load_json_file(path)
                if data:
                    report_data["analysis_files"][name] = {
                        "file": str(path),
                        "type": "json",
                        "summary": display_json_summary(data, max_keys=5)
                    }
            elif path.suffix == ".log":
                lines = load_log_file(path, max_lines=10)
                report_data["analysis_files"][name] = {
                    "file": str(path),
                    "type": "log",
                    "line_count": len(lines),
                    "preview": lines[:3] if lines else []
                }
    
    # Save report
    report_file = f"analysis_report_{timestamp}.json"
    return save_formatted_output(report_data, report_file, "json")

def create_code_snippet(code, language="python", description=""):
    """Create a reusable code snippet with metadata."""
    snippet = {
        "code": code,
        "language": language,
        "description": description,
        "created": pd.Timestamp.now().isoformat(),
        "repository": "NuSyQ-Hub"
    }
    return snippet

# Example usage functions
def demo_file_explorer():
    """Demonstrate the file explorer widget."""
    print("🔍 Interactive File Explorer:")
    return create_file_explorer()

def demo_json_explorer():
    """Demonstrate the JSON explorer widget."""
    print("📊 Interactive JSON Explorer:")
    return create_json_explorer()

def demo_repository_structure():
    """Demonstrate repository structure display."""
    print("🏗️ Repository Structure:")
    return display_repository_structure()

print("💾 Export and demo functions ready!")
print("\nNext steps:")
print("1. Run demo_file_explorer() to browse repository files")
print("2. Run demo_json_explorer() to explore JSON data")
print("3. Run demo_repository_structure() to view the full structure")
print("4. Run generate_analysis_report() to create a summary report")

💾 Export and demo functions ready!

Next steps:
1. Run demo_file_explorer() to browse repository files
2. Run demo_json_explorer() to explore JSON data
3. Run demo_repository_structure() to view the full structure
4. Run generate_analysis_report() to create a summary report


In [8]:
# Quick test: Load and display summary of a key JSON file
component_data = load_json_file(key_files["component_index"])
if component_data:
    print("📊 Component Index Summary:")
    print(display_json_summary(component_data, max_keys=5))
    print("\n" + "="*50 + "\n")

# Test the repository structure generation
print("🏗️ Testing Repository Structure Generation:")
try:
    structure_display = display_repository_structure()
    print("✅ Repository structure generated successfully!")
except Exception as e:
    print(f"❌ Error: {e}")

📊 Component Index Summary:
📊 JSON Summary (4 top-level keys)
   🗂️  metadata: dict with 4 items
   🗂️  components: dict with 157842 items
   🗂️  duplicates: dict with 892 items
   🗂️  categories: dict with 5 items


🏗️ Testing Repository Structure Generation:
✅ Repository structure generated successfully!


In [9]:
# Test widget functionality
print("🧪 Testing Interactive Widgets:")
print("Note: Run demo_file_explorer() or demo_json_explorer() in a new cell to see interactive widgets")

# Test DataFrame conversion
print("\n📊 Testing DataFrame Conversion:")
df = create_dataframe_from_json(component_data, "metadata")
if df is not None:
    print(f"✅ Successfully converted metadata to DataFrame: {df.shape}")
    print("First few columns:", list(df.columns[:5]))
else:
    print("⚠️ Could not convert to DataFrame")

# Test syntax highlighting on a small code sample
print("\n🎨 Testing Syntax Highlighting:")
sample_code = '''
def hello_world():
    """A simple greeting function."""
    return "Hello, NuSyQ-Hub!"

result = hello_world()
print(result)
'''

highlighted = highlight_code(sample_code, "python")
print("✅ Syntax highlighting generated successfully!")

🧪 Testing Interactive Widgets:
Note: Run demo_file_explorer() or demo_json_explorer() in a new cell to see interactive widgets

📊 Testing DataFrame Conversion:
✅ Successfully converted metadata to DataFrame: (1, 4)
First few columns: ['generated', 'repository_root', 'total_files_scanned', 'total_components_found']

🎨 Testing Syntax Highlighting:
✅ Syntax highlighting generated successfully!


## ✅ Notebook Setup Complete!

### 🎯 What's Ready:
- ✅ All libraries imported successfully
- ✅ Repository structure generator working
- ✅ JSON parsing and formatting functions ready
- ✅ Syntax highlighting operational
- ✅ Interactive widgets configured
- ✅ Export and analysis functions available

### 🚀 Next Steps - Try These Commands:

```python
# Interactive file browser
demo_file_explorer()

# Interactive JSON data explorer  
demo_json_explorer()

# Display full repository structure
demo_repository_structure()

# Generate comprehensive analysis report
generate_analysis_report()

# Load and explore specific files
data = load_json_file(key_files['dependency_map'])
display(display_json_summary(data))
```

### 📊 Available Repository Data:
- **Component Index**: 157,842 components catalogued
- **Dependency Mapping**: Complete project dependency analysis
- **Import Health**: Log file with import validation results  
- **File Organization**: Audit log of repository structure

### 💡 Pro Tips:
1. Use the interactive widgets to explore different file types
2. The syntax highlighter supports auto-detection of languages
3. Export functions can save your analysis results
4. All functions include error handling for robust operation